# Problem 1 — RNN, LSTM and GRU

Set the individual variant values below from the instructor's table. The supplied document refers to that table, but it was not included. Complete the explicitly marked scratch-cell implementation before running comparison tests. Do not submit fabricated metrics.

In [1]:
from pathlib import Path
import sys
import numpy as np
import torch

ROOT = Path.cwd()
if not (ROOT / 'problem1').is_dir():
    if (ROOT / 'assignment' / 'problem1').is_dir():
        ROOT = ROOT / 'assignment'
    elif (ROOT.parent / 'problem1').is_dir():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from problem1.data import make_adding, assert_adding_data, load_har
from problem1.models import RecurrentClassifier, RecurrentRegressor, count_parameters
from problem1.train import set_seed, make_loader, train_classifier, evaluate_classifier, train_regressor, predict_mse
from problem1.experiments import (
    parameter_count_reference,
    plot_har_class_balance,
    run_adding_trials,
    run_har_comparison,
    run_hidden_size_sweep,
    run_selected_ablation,
)

# Provisional values only: replace using the instructor's individualisation table.
HIDDEN_SIZE = 64
SEEDS = [11, 21, 31]
BATCH_SIZE = 128
EPOCHS = 10
ADDING_EPOCHS = 20
ADDING_TRAIN_SIZE = 5000
ADDING_VAL_SIZE = 1000
ADDING_TEST_SIZE = 2000
STUDENT_ID_LAST_DIGIT = 3
HAR_ROOT = ROOT / 'problem1/data/UCI HAR Dataset'
RESULTS_PATH = ROOT / 'problem1/results.csv'
FIGURE_DIR = ROOT / 'problem1/figures'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cpu


## Adding-problem generator check

The checks are adjacent to the generator invocation as required. Repeat with all requested sequence lengths and data splits.

In [2]:
for sequence_length in (50, 100, 200, 400):
    for seed in SEEDS:
        x_check, y_check = make_adding(
            n=512, sequence_length=sequence_length, seed=seed
        )
        assert_adding_data(x_check, y_check)
        baseline_mse = float(np.mean((y_check - 1.0) ** 2))
        print(
            f'T={sequence_length}, seed={seed}, shape={x_check.shape}, '
            f'baseline MSE={baseline_mse:.4f}'
        )

T=50, seed=11, shape=(512, 50, 2), baseline MSE=0.1691
T=50, seed=21, shape=(512, 50, 2), baseline MSE=0.1638
T=50, seed=31, shape=(512, 50, 2), baseline MSE=0.1585
T=100, seed=11, shape=(512, 100, 2), baseline MSE=0.1578
T=100, seed=21, shape=(512, 100, 2), baseline MSE=0.1735
T=100, seed=31, shape=(512, 100, 2), baseline MSE=0.1691
T=200, seed=11, shape=(512, 200, 2), baseline MSE=0.1652
T=200, seed=21, shape=(512, 200, 2), baseline MSE=0.1554
T=200, seed=31, shape=(512, 200, 2), baseline MSE=0.1611
T=400, seed=11, shape=(512, 400, 2), baseline MSE=0.1545
T=400, seed=21, shape=(512, 400, 2), baseline MSE=0.1723
T=400, seed=31, shape=(512, 400, 2), baseline MSE=0.1659


## HAR data and class balance

Use one validation-subject split per prescribed seed. Plot training class counts here and report the chosen best model's test confusion matrix.

In [3]:
har = load_har(HAR_ROOT, seed=SEEDS[0])
print({key: value.shape for key, value in har.items() if isinstance(value, np.ndarray)})
print('train subjects:', har['train_subjects'], 'validation subjects:', har['val_subjects'])
class_balance_figure = plot_har_class_balance(har['y_train'], FIGURE_DIR)
print('training class balance saved to:', class_balance_figure)

{'x_train': (5891, 128, 9), 'y_train': (5891,), 'x_val': (1461, 128, 9), 'y_val': (1461,), 'x_test': (2947, 128, 9), 'y_test': (2947,), 'train_subjects': (17,), 'val_subjects': (4,), 'mean': (1, 1, 9), 'std': (1, 1, 9)}
train subjects: [ 1  3  6  7  8 11 14 15 16 19 21 22 23 26 27 29 30] validation subjects: [ 5 17 25 28]
training class balance saved to: c:\Users\balaj\IdeaProjects\DNN\DNN_Assignment_Starter\problem1\figures\har_train_class_balance.png


## Scratch recurrent cells and equivalence checks

Implement `MyRNNCell`, `MyLSTMCell`, and `MyGRUCell` in `from_scratch.py` yourself. Copy each library layer's parameters into your implementation and compare sequence outputs to a random batch with maximum absolute error below 1e-5. Include state-output checks as well as the required parameter-count derivations.

In [4]:
from problem1 import from_scratch
from torch import nn

print('Cell parameter counts (one bias vs PyTorch two biases):')
reference_counts = parameter_count_reference(9, 64)
for cell_name, counts in reference_counts.items():
    print(f"{cell_name}: {counts['one_bias']:,} vs {counts['two_bias']:,}")


def copy_layer_parameters(reference, scratch):
    with torch.no_grad():
        for name in ('weight_ih', 'weight_hh', 'bias_ih', 'bias_hh'):
            getattr(scratch, name).copy_(getattr(reference, f'{name}_l0'))


torch.manual_seed(SEEDS[0])
batch_size, sequence_length, input_size, hidden_size = 4, 17, 9, 64
x_equivalence = torch.randn(batch_size, sequence_length, input_size, device=DEVICE)
h0 = torch.zeros(1, batch_size, hidden_size, device=DEVICE)
c0 = torch.zeros_like(h0)

reference_rnn = nn.RNN(input_size, hidden_size, batch_first=True).to(DEVICE)
scratch_rnn = from_scratch.MyRNNCell(input_size, hidden_size).to(DEVICE)
copy_layer_parameters(reference_rnn, scratch_rnn)
h = h0[0]
rnn_outputs = []
for time_index in range(sequence_length):
    h = scratch_rnn(x_equivalence[:, time_index], h)
    rnn_outputs.append(h)
rnn_sequence = torch.stack(rnn_outputs, dim=1)
rnn_reference_sequence, rnn_reference_state = reference_rnn(x_equivalence, h0)
rnn_error = max(
    (rnn_sequence - rnn_reference_sequence).abs().max().item(),
    (h - rnn_reference_state[0]).abs().max().item(),
)

reference_lstm = nn.LSTM(input_size, hidden_size, batch_first=True).to(DEVICE)
scratch_lstm = from_scratch.MyLSTMCell(input_size, hidden_size).to(DEVICE)
copy_layer_parameters(reference_lstm, scratch_lstm)
h, c = h0[0], c0[0]
lstm_outputs = []
for time_index in range(sequence_length):
    h, c = scratch_lstm(x_equivalence[:, time_index], (h, c))
    lstm_outputs.append(h)
lstm_sequence = torch.stack(lstm_outputs, dim=1)
lstm_reference_sequence, (lstm_reference_h, lstm_reference_c) = reference_lstm(
    x_equivalence, (h0, c0)
)
lstm_error = max(
    (lstm_sequence - lstm_reference_sequence).abs().max().item(),
    (h - lstm_reference_h[0]).abs().max().item(),
    (c - lstm_reference_c[0]).abs().max().item(),
)

reference_gru = nn.GRU(input_size, hidden_size, batch_first=True).to(DEVICE)
scratch_gru = from_scratch.MyGRUCell(input_size, hidden_size).to(DEVICE)
copy_layer_parameters(reference_gru, scratch_gru)
h = h0[0]
gru_outputs = []
for time_index in range(sequence_length):
    h = scratch_gru(x_equivalence[:, time_index], h)
    gru_outputs.append(h)
gru_sequence = torch.stack(gru_outputs, dim=1)
gru_reference_sequence, gru_reference_state = reference_gru(x_equivalence, h0)
gru_error = max(
    (gru_sequence - gru_reference_sequence).abs().max().item(),
    (h - gru_reference_state[0]).abs().max().item(),
)

assert max(rnn_error, lstm_error, gru_error) < 1e-5
assert sum(p.numel() for p in scratch_rnn.parameters()) == reference_counts['RNN']['two_bias']
assert sum(p.numel() for p in scratch_lstm.parameters()) == reference_counts['LSTM']['two_bias']
assert sum(p.numel() for p in scratch_gru.parameters()) == reference_counts['GRU']['two_bias']
print(f'max absolute errors: RNN={rnn_error:.2e}, LSTM={lstm_error:.2e}, GRU={gru_error:.2e}')

Cell parameter counts (one bias vs PyTorch two biases):
RNN: 4,736 vs 4,800
LSTM: 18,944 vs 19,200
GRU: 14,208 vs 14,400
max absolute errors: RNN=0.00e+00, LSTM=5.96e-08, GRU=6.71e-08


## HAR comparisons and adding experiments

Use the same optimizer, batch size, epoch budget, hidden size, and each of the three prescribed seeds for RNN/LSTM/GRU. Report mean and sample standard deviation; append one row per run to `results.csv`. Add the required plots and ablations from the assignment. The helper functions below are intended for these experiments.

In [5]:
# This cell runs the measured experiments and appends one CSV row per run.
# Full runs may be lengthy, especially the hidden-size sweep and T=400 trials.
har_records, best_har = run_har_comparison(
    HAR_ROOT,
    SEEDS,
    HIDDEN_SIZE,
    BATCH_SIZE,
    EPOCHS,
    DEVICE,
    RESULTS_PATH,
    FIGURE_DIR,
)
hidden_sweep_records = run_hidden_size_sweep(
    HAR_ROOT,
    SEEDS,
    (16, 32, 64, 128),
    BATCH_SIZE,
    EPOCHS,
    DEVICE,
    RESULTS_PATH,
)
adding_records, gradient_norms = run_adding_trials(
    SEEDS,
    (50, 100, 200, 400),
    HIDDEN_SIZE,
    BATCH_SIZE,
    ADDING_EPOCHS,
    ADDING_TRAIN_SIZE,
    ADDING_VAL_SIZE,
    ADDING_TEST_SIZE,
    DEVICE,
    RESULTS_PATH,
    FIGURE_DIR,
)
if STUDENT_ID_LAST_DIGIT is None:
    print('Set STUDENT_ID_LAST_DIGIT from the instructor mapping, then run the ablation.')
else:
    ablation_records = run_selected_ablation(
        STUDENT_ID_LAST_DIGIT,
        SEEDS,
        HIDDEN_SIZE,
        BATCH_SIZE,
        ADDING_EPOCHS,
        ADDING_TRAIN_SIZE,
        ADDING_VAL_SIZE,
        ADDING_TEST_SIZE,
        DEVICE,
        RESULTS_PATH,
    )

HAR RNN: accuracy=0.5941 +/- 0.0544, macro-F1=0.5312 +/- 0.0785, params=5190, sec/epoch=1.498 +/- 0.041
HAR LSTM: accuracy=0.8303 +/- 0.0341, macro-F1=0.8262 +/- 0.0357, params=19590, sec/epoch=1.679 +/- 0.072
HAR GRU: accuracy=0.8782 +/- 0.0072, macro-F1=0.8781 +/- 0.0059, params=14790, sec/epoch=4.956 +/- 0.156
Best validation run test confusion matrix saved to c:\Users\balaj\IdeaProjects\DNN\DNN_Assignment_Starter\problem1\figures\har_best_confusion_matrix.png
HAR sweep LSTM h=16: accuracy=0.7112 +/- 0.0376, macro-F1=0.6914 +/- 0.0449, params=1830
HAR sweep LSTM h=32: accuracy=0.7858 +/- 0.0258, macro-F1=0.7799 +/- 0.0317, params=5702
HAR sweep LSTM h=64: accuracy=0.8303 +/- 0.0341, macro-F1=0.8262 +/- 0.0357, params=19590
HAR sweep LSTM h=128: accuracy=0.8766 +/- 0.0077, macro-F1=0.8769 +/- 0.0061, params=71942
HAR sweep GRU h=16: accuracy=0.7837 +/- 0.0255, macro-F1=0.7784 +/- 0.0285, params=1398
HAR sweep GRU h=32: accuracy=0.8698 +/- 0.0286, macro-F1=0.8698 +/- 0.0271, params=43